# 16 OpenTripMap Expanded Collection

Collect a broader tourism-focused OpenTripMap POI set for Istanbul using manual landmark-center coordinates.

In [41]:
import os
import time
import requests
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)


In [42]:
LANG = "en"
COUNTRY = "TR"
RADIUS_METERS = 1400
LIMIT_PER_AREA = 40
DETAIL_LIMIT = 20
KIND_FILTER = "museums,historic_architecture,monuments,architecture,churches,mosques,interesting_places"

OUTPUT_PREFIX = "../data/processed/opentripmap_expanded"

TARGET_CENTERS = [
    {"query_name": "Sultanahmet Core", "lat": 41.0054, "lon": 28.9768},
    {"query_name": "Hagia Sophia / Basilica Cistern", "lat": 41.0080, "lon": 28.9799},
    {"query_name": "Topkapi Palace", "lat": 41.0115, "lon": 28.9833},
    {"query_name": "Eminonu / Spice Bazaar", "lat": 41.0165, "lon": 28.9702},
    {"query_name": "Suleymaniye", "lat": 41.0162, "lon": 28.9638},
    {"query_name": "Fatih / Valens Aqueduct", "lat": 41.0170, "lon": 28.9565},
    {"query_name": "Galata Tower", "lat": 41.0257, "lon": 28.9744},
    {"query_name": "Istiklal / Pera", "lat": 41.0339, "lon": 28.9779},
    {"query_name": "Dolmabahce", "lat": 41.0392, "lon": 29.0007},
    {"query_name": "Ortakoy", "lat": 41.0473, "lon": 29.0260},
    {"query_name": "Uskudar Waterfront", "lat": 41.0258, "lon": 29.0152},
    {"query_name": "Maiden's Tower Coast", "lat": 41.0211, "lon": 29.0041},
    {"query_name": "Kadikoy Historic Center", "lat": 40.9909, "lon": 29.0263},
    {"query_name": "Balat / Fener", "lat": 41.0297, "lon": 28.9497},
    {"query_name": "Beylerbeyi Palace", "lat": 41.0420, "lon": 29.0405},
    {"query_name": "Rumeli Hisari", "lat": 41.0849, "lon": 29.0568},
    {"query_name": "Eyup / Pierre Loti", "lat": 41.0480, "lon": 28.9327},
    {"query_name": "Yildiz Palace", "lat": 41.0438, "lon": 29.0125},
]

OPENTRIPMAP_API_KEY = os.getenv("OPENTRIPMAP_API_KEY")
if not OPENTRIPMAP_API_KEY:
    OPENTRIPMAP_API_KEY = input("Paste OpenTripMap API key: ").strip()

print("Key loaded:", bool(OPENTRIPMAP_API_KEY))
print("Target centers:", len(TARGET_CENTERS))


Key loaded: True
Target centers: 18


## Helpers

In [43]:
BASE_URL = f"https://api.opentripmap.com/0.1/{LANG}/places"


def otm_get(method, params=None):
    if not OPENTRIPMAP_API_KEY:
        raise ValueError("OPENTRIPMAP_API_KEY is missing")

    query = dict(params or {})
    query["apikey"] = OPENTRIPMAP_API_KEY

    url = f"{BASE_URL}/{method}"
    response = requests.get(url, params=query, timeout=20)

    if response.status_code != 200:
        debug = {
            "status_code": response.status_code,
            "response_text": response.text[:500],
            "url": response.url,
        }
        print("OpenTripMap error debug:", debug)
        response.raise_for_status()

    return response.json()


def get_places_radius(lat, lon, radius=1000, limit=20, kinds=None, rate=2, fmt="json"):
    params = {
        "radius": radius,
        "lon": lon,
        "lat": lat,
        "limit": limit,
        "rate": rate,
        "format": fmt,
    }
    if kinds:
        params["kinds"] = kinds
    return otm_get("radius", params)


def get_place_details(xid):
    return otm_get(f"xid/{xid}")


## Step 1 - Collection centers

In [44]:
centers_df = pd.DataFrame(TARGET_CENTERS)
centers_df


,query_name,lat,lon
0,Sultanahmet Core,41.0054,28.9768
1,Hagia Sophia / Basilica Cistern,41.0080,28.9799
2,Topkapi Palace,41.0115,28.9833
3,Eminonu / Spice Bazaar,41.0165,28.9702
4,Suleymaniye,41.0162,28.9638
5,Fatih / Valens Aqueduct,41.0170,28.9565
6,Galata Tower,41.0257,28.9744
7,Istiklal / Pera,41.0339,28.9779
8,Dolmabahce,41.0392,29.0007
9,Ortakoy,41.0473,29.0260


## Step 2 - Pull nearby POIs

In [45]:
radius_frames = []

for _, center_row in centers_df.iterrows():
    radius_results = get_places_radius(
        lat=center_row["lat"],
        lon=center_row["lon"],
        radius=RADIUS_METERS,
        limit=LIMIT_PER_AREA,
        kinds=KIND_FILTER,
    )

    area_df = pd.DataFrame(radius_results)
    if not area_df.empty:
        area_df["query_area"] = center_row["query_name"]
        radius_frames.append(area_df)
    time.sleep(0.5)

radius_df = pd.concat(radius_frames, ignore_index=True) if radius_frames else pd.DataFrame()
if not radius_df.empty and "xid" in radius_df.columns:
    radius_df = radius_df.sort_values(["rate", "dist"], ascending=[False, True])
    radius_df = radius_df.drop_duplicates(subset=["xid"]).reset_index(drop=True)

radius_df.head(30)


,xid,name,dist,rate,osm,wikidata,kinds,point,query_area
0,N7215645385,The Blue Mosque,18.136162,7,node/7215645385,Q80541,"religion,mosques,interesting_places","{'lon': 28.976892471313477, 'lat': 41.00525283...",Sultanahmet Core
1,R1564032,Süleymaniye Mosque,18.644092,7,relation/1564032,Q178643,"religion,mosques,interesting_places","{'lon': 28.963977813720703, 'lat': 41.01630020...",Suleymaniye
2,Q5773394,Historic Areas of Istanbul,52.256390,7,NaN,Q5773394,"interesting_places,natural,nature_reserves,oth...","{'lon': 28.979930877685547, 'lat': 41.00846862...",Hagia Sophia / Basilica Cistern
3,R1555271,Hagia Sophia Grand Mosque,57.237152,7,relation/1555271,Q12506,"religion,mosques,churches,cultural,museums,int...","{'lon': 28.980009078979492, 'lat': 41.00850677...",Hagia Sophia / Basilica Cistern
4,W109862851,The Hagia Sophia Grand Mosque,57.237152,7,way/109862851,Q12506,"religion,mosques,churches,cultural,museums,int...","{'lon': 28.980009078979492, 'lat': 41.00850677...",Hagia Sophia / Basilica Cistern
5,Q2656937,Runic inscriptions in Hagia Sophia,102.822256,7,NaN,Q2656937,"other,unclassified_objects,interesting_places,...","{'lon': 28.97909927368164, 'lat': 41.008701324...",Hagia Sophia / Basilica Cistern
6,N415157636,Serpent Column,145.719100,7,node/415157636,Q588892,"historic,monuments_and_memorials,burial_places...","{'lon': 28.9751033782959, 'lat': 41.0056610107...",Sultanahmet Core
7,W103953125,Tomb of Sultan Ahmet,155.641087,7,way/103953125,Q80541,"religion,mosques,interesting_places","{'lon': 28.97703742980957, 'lat': 41.006790161...",Sultanahmet Core
8,W326372295,Yıldız Palace,700.923972,7,way/326372295,Q911734,"palaces,architecture,historic_architecture,cul...","{'lon': 29.01161766052246, 'lat': 41.050075531...",Yildiz Palace
9,N4477143891,Yıldız Palace,770.380137,7,node/4477143891,Q911734,"palaces,architecture,historic_architecture,int...","{'lon': 29.011703491210938, 'lat': 41.05071258...",Yildiz Palace


In [46]:
if not radius_df.empty:
    display_cols = [c for c in ["query_area", "xid", "name", "kinds", "dist", "rate", "osm", "wikidata"] if c in radius_df.columns]
    print("Radius result shape:", radius_df.shape)
    print(radius_df[display_cols].head(40).to_string())
else:
    print("No radius results returned.")


Radius result shape: (395, 9)
                         query_area          xid                                name                                                                                                                                  kinds        dist  rate               osm    wikidata
0                  Sultanahmet Core  N7215645385                     The Blue Mosque                                                                                                    religion,mosques,interesting_places   18.136162     7   node/7215645385      Q80541
1                       Suleymaniye     R1564032                  Süleymaniye Mosque                                                                                                    religion,mosques,interesting_places   18.644092     7  relation/1564032     Q178643
2   Hagia Sophia / Basilica Cistern     Q5773394          Historic Areas of Istanbul                                                             interesting_places,na

## Step 3 - Pull details for a controlled subset

In [47]:
detail_xids = radius_df["xid"].dropna().head(DETAIL_LIMIT).tolist() if "xid" in radius_df.columns else []
detail_xids


['N7215645385',
 'R1564032',
 'Q5773394',
 'R1555271',
 'W109862851',
 'Q2656937',
 'N415157636',
 'W103953125',
 'W326372295',
 'N4477143891',
 'N7294561685',
 'W32395058',
 'R7318154',
 'W102190099',
 'W23236783',
 'N6919989286',
 'Q105840441',
 'N4519290891',
 'W261825033',
 'N269370709']

In [48]:
detail_rows = []
area_lookup = radius_df[["xid", "query_area"]].drop_duplicates().set_index("xid")["query_area"].to_dict() if not radius_df.empty else {}

for xid in detail_xids:
    try:
        details = get_place_details(xid)
        details["query_area"] = area_lookup.get(xid)
        detail_rows.append(details)
        time.sleep(0.5)
    except Exception as exc:
        print(f"Skipping {xid}: {exc}")

details_df = pd.DataFrame(detail_rows)
details_df.head(20)


,xid,name,address,rate,osm,wikidata,kinds,url,sources,otm,wikipedia,image,preview,wikipedia_extracts,point,query_area,bbox
0,N7215645385,The Blue Mosque,"{'town': 'Fatih', 'house': 'Sultanahmet Camii'...",3h,node/7215645385,Q80541,"religion,mosques,interesting_places",https://www.geziyerler.com/sultan-ahmet-camii;...,"{'geometry': 'osm', 'attributes': ['osm', 'wik...",https://opentripmap.com/en/card/N7215645385,https://en.wikipedia.org/wiki/Sultan%20Ahmed%2...,https://commons.wikimedia.org/wiki/File:Sultan...,{'source': 'https://upload.wikimedia.org/wikip...,"{'title': 'en:Sultan Ahmed Mosque', 'text': 'S...","{'lon': 28.976892471313477, 'lat': 41.00525283...",Sultanahmet Core,NaN
1,R1564032,Süleymaniye Mosque,"{'city': 'Süleymaniye Mahallesi', 'road': 'Pro...",3h,relation/1564032,Q178643,"religion,mosques,interesting_places",https://www.geziyerler.com/suleymaniye-camii,"{'geometry': 'osm', 'attributes': ['osm', 'wik...",https://opentripmap.com/en/card/R1564032,https://en.wikipedia.org/wiki/S%C3%BCleymaniye...,https://commons.wikimedia.org/wiki/File:Istanb...,{'source': 'https://upload.wikimedia.org/wikip...,"{'title': 'en:Süleymaniye Mosque', 'text': 'Th...","{'lon': 28.963977813720703, 'lat': 41.01630020...",Suleymaniye,"{'lon_min': 28.963088, 'lon_max': 28.964576, '..."
2,Q5773394,Historic Areas of Istanbul,"{'city': 'Cankurtaran Mahallesi', 'town': 'Fat...",3h,NaN,Q5773394,"interesting_places,natural,nature_reserves,oth...",NaN,"{'geometry': 'wikidata', 'attributes': ['wikid...",https://opentripmap.com/en/card/Q5773394,https://en.wikipedia.org/wiki/Historic%20Areas...,https://commons.wikimedia.org/wiki/File:Istanb...,{'source': 'https://upload.wikimedia.org/wikip...,"{'title': 'en:Historic Areas of Istanbul', 'te...","{'lon': 28.979930877685547, 'lat': 41.00846862...",Hagia Sophia / Basilica Cistern,NaN
3,R1555271,Hagia Sophia Grand Mosque,"{'city': 'Cankurtaran Mahallesi', 'road': 'Bab...",3h,relation/1555271,Q12506,"religion,mosques,churches,cultural,museums,int...",https://muze.gen.tr/muze-detay/ayasofya;https:...,"{'geometry': 'osm', 'attributes': ['osm', 'wik...",https://opentripmap.com/en/card/R1555271,https://en.wikipedia.org/wiki/Hagia%20Sophia,https://commons.wikimedia.org/wiki/File:Hagia_...,{'source': 'https://upload.wikimedia.org/wikip...,"{'title': 'en:Hagia Sophia', 'text': 'Hagia So...","{'lon': 28.980009078979492, 'lat': 41.00850677...",Hagia Sophia / Basilica Cistern,"{'lon_min': 28.978778, 'lon_max': 28.980948, '..."
4,W109862851,The Hagia Sophia Grand Mosque,"{'city': 'İstanbul', 'town': 'Fatih', 'state':...",3h,way/109862851,Q12506,"religion,mosques,churches,cultural,museums,int...",https://muze.gen.tr/muze-detay/ayasofya;https:...,"{'geometry': 'osm', 'attributes': ['osm', 'wik...",https://opentripmap.com/en/card/W109862851,https://en.wikipedia.org/wiki/Hagia%20Sophia,https://commons.wikimedia.org/wiki/File:Hagia_...,{'source': 'https://upload.wikimedia.org/wikip...,"{'title': 'en:Hagia Sophia', 'text': 'Hagia So...","{'lon': 28.980009078979492, 'lat': 41.00850677...",Hagia Sophia / Basilica Cistern,"{'lon_min': 28.979078, 'lon_max': 28.980917, '..."
5,Q2656937,Runic inscriptions in Hagia Sophia,"{'city': 'Cankurtaran Mahallesi', 'road': 'Caf...",3h,NaN,Q2656937,"other,unclassified_objects,interesting_places,...",NaN,"{'geometry': 'wikidata', 'attributes': ['wikid...",https://opentripmap.com/en/card/Q2656937,https://en.wikipedia.org/wiki/Runic%20inscript...,https://commons.wikimedia.org/wiki/File:Hagia-...,{'source': 'https://upload.wikimedia.org/wikip...,{'title': 'en:Runic inscriptions in Hagia Soph...,"{'lon': 28.97909927368164, 'lat': 41.008701324...",Hagia Sophia / Basilica Cistern,NaN
6,N415157636,Serpent Column,"{'city': 'Binbirdirek Mahallesi', 'town': 'Fat...",3h,node/415157636,Q588892,"historic,monuments_and_memorials,burial_places...",NaN,"{'geometry': 'osm', 'attributes': ['osm', 'wik...",https://opentripmap.com/en/card/N415157636,https://en.wikipedia.org/wiki/Serpent%20Column,https://commo

## Save outputs

In [49]:
centers_df.to_csv(f"{OUTPUT_PREFIX}_centers.csv", index=False)
print(f"Saved: {OUTPUT_PREFIX}_centers.csv")

if radius_df.empty:
    print("Skipping save because no radius results were returned.")
else:
    radius_df.to_csv(f"{OUTPUT_PREFIX}_radius_results.csv", index=False)
    print(f"Saved: {OUTPUT_PREFIX}_radius_results.csv")

if 'details_df' in globals() and not details_df.empty:
    details_df.to_csv(f"{OUTPUT_PREFIX}_details_results.csv", index=False)
    print(f"Saved: {OUTPUT_PREFIX}_details_results.csv")
else:
    print("Skipping details save because no detail rows were collected.")


Saved: ../data/processed/opentripmap_expanded_centers.csv
Saved: ../data/processed/opentripmap_expanded_radius_results.csv
Saved: ../data/processed/opentripmap_expanded_details_results.csv


## Notes

- This expanded version uses manual tourism centers instead of area-name geocoding.
- Expanded outputs are written to `opentripmap_expanded_*` files so the smaller pilot outputs remain intact.
- If the expanded pull becomes too noisy, tighten `KIND_FILTER` or reduce `RADIUS_METERS`.
- If the expanded pull is still too small, increase `LIMIT_PER_AREA` before increasing `DETAIL_LIMIT`.